# Анализ текстовых моделей / Text Models Analysis

Ноутбук загружает обученные текстовые модели классификации эмоций речи и оценивает их
на тестовом наборе данных **Dusha + resd** (`dusha_resd_test.lmdb`).

Для каждой модели выводятся:
- Сводные метрики (Accuracy, Balanced Accuracy, F1, MCC, ROC-AUC)
- Отчёт по классам (Precision / Recall / F1)
- Матрица ошибок (confusion matrix)

**Эмоции (4 класса):** `angry` · `sad` · `neutral` · `positive`

> Используются сторонние предобученные модели; их источники и лицензии — [SOURCES.md](../../../SOURCES.md).


---
## Содержание

| # | Модель | Описание |
|---|--------|----------|
| 1 | [TF-IDF + LogReg](#1-tf-idf--logistic-regression) | Bag-of-words + линейный классификатор |
| 2 | [FastText + LogReg](#2-fasttext-embeddings--logistic-regression) | Усреднённые word embeddings + линейный классификатор |
| 3 | [BiLSTM](#3-bilstm-word-level-fasttext-embeddings) | Рекуррентная нейросеть на уровне слов |
| 4 | [RuBERT](#4-rubert-deeppavlov) | Предобученный трансформер (DeepPavlov) |

In [ ]:
import sys
from pathlib import Path

def _find_repo_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'ruintona' / 'my_experiments').is_dir():
            return p
    return None

_REPO_ROOT = _find_repo_root()
if _REPO_ROOT is not None and str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from ruintona.my_experiments.utils.config_utils import (
    PROJECT_ROOT, DATASET_PATH, TARGET_NAMES, EMO2LABEL,
)

import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

AGGREGATED_DIR = DATASET_PATH / 'processed_dataset_090' / 'aggregated_dataset'

# TEST DATASETS - uncomment one:
# TEST_LMDB = AGGREGATED_DIR / 'combine_balanced_test.lmdb'
# TEST_LMDB = AGGREGATED_DIR / 'combine_balanced_test_small.lmdb'
TEST_LMDB = AGGREGATED_DIR / 'dusha_resd_test.lmdb'

CHECKPOINTS_DIR = PROJECT_ROOT / 'my_experiments' / 'checkpoints' / 'text'
PRETRAINED_DIR = PROJECT_ROOT / 'my_experiments' / 'checkpoints' / 'pretrained'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Test dataset: {TEST_LMDB}')
print(f'Exists: {TEST_LMDB.exists()}')
print(f'Device: {DEVICE}')
print(f'Checkpoints: {CHECKPOINTS_DIR}')

---
## Helper functions

In [ ]:
from ruintona.my_experiments.utils.model_io import load_sklearn_model, load_pytorch_model
from ruintona.my_experiments.utils.text_utils import load_texts_from_manifest, preprocess_text
from ruintona.my_experiments.utils.metrics import print_eval_block, compute_classification_metrics
from ruintona.my_experiments.utils.lmdb_utils import parse_label_to_index
from ruintona.my_experiments.utils.config_utils import TARGET_NAMES


def load_texts(lmdb_path):
    texts, labels = load_texts_from_manifest(lmdb_path)
    labels_num = np.array([EMO2LABEL[l] for l in labels])
    return texts, labels, labels_num


def texts_to_vectors(texts, ft_model):
    vectors = []
    for text in texts:
        words = text.split()
        word_vecs = [ft_model.wv[w] for w in words if w in ft_model.wv]
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(np.zeros(ft_model.wv.vector_size))
    return np.array(vectors)

---
## 1. TF-IDF + Logistic Regression

**Type:** Bag-of-words + linear model  
**Features:** TF-IDF unigrams  
**Preprocessing:** lowercase, collapse whitespace  
**Checkpoint:** `TF-IDF_LogReg_combine_balanced_train_model.pkl` + `_vectorizer.pkl`

In [ ]:
dataset_name = 'combine_balanced_train'

model_tfidf, vectorizer = load_sklearn_model(
    dataset_name, models_dir=CHECKPOINTS_DIR,
    model_name='TF-IDF_LogReg', artifact_name='vectorizer'
)

texts_test, labels_test_str, labels_test_num = load_texts(TEST_LMDB)
X_test_tfidf = vectorizer.transform(texts_test)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)
y_pred_tfidf_num = np.array([EMO2LABEL[p] for p in y_pred_tfidf])

has_proba = hasattr(model_tfidf, 'predict_proba')
probs_tfidf = model_tfidf.predict_proba(X_test_tfidf) if has_proba else None

metrics_tfidf = compute_classification_metrics(labels_test_num, y_pred_tfidf_num, probs_tfidf)
print_eval_block('TF-IDF + LogReg - Test Metrics', metrics_tfidf, labels_test_num, y_pred_tfidf_num)

---
## 2. FastText Embeddings + Logistic Regression

**Type:** Averaged word embeddings + linear model  
**Features:** FastText 300d (mean pooling) + StandardScaler  
**Preprocessing:** lowercase, collapse whitespace  
**Checkpoint:** `Embeddings_LogReg_combine_balanced_train_model.pkl` + `_scaler.pkl`  
**External embeddings:** `pretrained/fasttext/cc.ru.300.bin`

In [ ]:
from ruintona.my_experiments.utils.text_utils import load_fasttext_model

dataset_name = 'combine_balanced_train'

model_emb, scaler_emb = load_sklearn_model(
    dataset_name, models_dir=CHECKPOINTS_DIR, model_name='Embeddings_LogReg'
)

fasttext_path = PRETRAINED_DIR / 'fasttext' / 'cc.ru.300.bin'
if not fasttext_path.exists():
    raise FileNotFoundError(f'FastText model not found: {fasttext_path}')

fasttext_model = load_fasttext_model(fasttext_path)

texts_test_emb, _, labels_test_num_emb = load_texts(TEST_LMDB)
X_test_emb = texts_to_vectors(texts_test_emb, fasttext_model)
X_test_emb_scaled = scaler_emb.transform(X_test_emb)
y_pred_emb_str = model_emb.predict(X_test_emb_scaled)
y_pred_emb = np.array([EMO2LABEL[p] for p in y_pred_emb_str])

probs_emb = model_emb.predict_proba(X_test_emb_scaled) if hasattr(model_emb, 'predict_proba') else None

metrics_emb = compute_classification_metrics(labels_test_num_emb, y_pred_emb, probs_emb)
print_eval_block('Embeddings + LogReg - Test Metrics', metrics_emb, labels_test_num_emb, y_pred_emb)

---
## 3. BiLSTM (word-level, FastText embeddings)

**Type:** Recurrent neural network  
**Architecture:** Embedding (FastText) → BiLSTM → Pooling → Linear  
**Tokenization:** Whitespace, pre-built vocabulary  
**Preprocessing:** lowercase, collapse whitespace  
**Checkpoint:** `BiLSTM_combine_balanced_train_model.pt`

In [ ]:
import re
from torch import nn
from torch.utils.data import DataLoader, Dataset

from ruintona.my_experiments.text_models.BiLSTM.BiLSTM import BiLSTMEmotionClassifier


class TextSequenceDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        unk_idx = word2idx.get('<UNK>', 1)
        self.sequences = []
        self.labels = []
        for text, label in zip(texts, labels):
            tokens = text.lower().split()[:max_len]
            seq = [word2idx.get(t, unk_idx) for t in tokens]
            self.sequences.append(torch.tensor(seq, dtype=torch.long))
            self.labels.append(torch.tensor(label, dtype=torch.long))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate_bilstm(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, torch.stack(labels), lengths


checkpoint_bl = load_pytorch_model(
    'combine_balanced_train', models_dir=CHECKPOINTS_DIR,
    model_name='BiLSTM', map_location=DEVICE
)

word2idx = checkpoint_bl.get('word2idx', {})
if not word2idx and 'extra_artifacts' in checkpoint_bl:
    word2idx = checkpoint_bl['extra_artifacts'].get('word2idx.json', {})

embedding_matrix = checkpoint_bl.get('embedding_matrix', None)
if embedding_matrix is None and 'extra_artifacts' in checkpoint_bl:
    embedding_matrix = checkpoint_bl['extra_artifacts'].get('embedding_matrix.pkl')

model_params_bl = checkpoint_bl.get('model_params', {})
max_len_bl = int(model_params_bl.get('max_len', 64))

model_bl = BiLSTMEmotionClassifier(
    embedding_matrix=embedding_matrix,
    hidden_size=model_params_bl.get('hidden_size', 256),
    num_layers=model_params_bl.get('num_layers', 2),
    dropout=model_params_bl.get('dropout', 0.3),
    freeze_embeddings=model_params_bl.get('freeze_embeddings', False),
    n_classes=model_params_bl.get('n_classes', 4),
    pooling_mode=model_params_bl.get('pooling_mode', 'mean_max'),
)
model_bl.load_state_dict(checkpoint_bl['model_state_dict'])
model_bl.to(DEVICE)
model_bl.eval()

texts_test_bl, _, labels_test_num_bl = load_texts(TEST_LMDB)
test_ds_bl = TextSequenceDataset(texts_test_bl, labels_test_num_bl, word2idx, max_len_bl)
test_loader_bl = DataLoader(test_ds_bl, batch_size=64, shuffle=False, collate_fn=collate_bilstm)

criterion = nn.CrossEntropyLoss()
all_preds_bl, all_targets_bl, all_probs_bl = [], [], []
running_loss_bl = 0.0
with torch.no_grad():
    for input_ids, labels, lengths in test_loader_bl:
        input_ids, labels, lengths = input_ids.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
        logits = model_bl(input_ids, lengths)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        running_loss_bl += loss.item() * input_ids.size(0)
        all_preds_bl.append(preds.cpu().numpy())
        all_targets_bl.append(labels.cpu().numpy())
        all_probs_bl.append(probs.cpu().numpy())

y_pred_bl = np.concatenate(all_preds_bl)
y_true_bl = np.concatenate(all_targets_bl)
probs_bl = np.concatenate(all_probs_bl)
metrics_bl = compute_classification_metrics(y_true_bl, y_pred_bl, probs_bl)
metrics_bl['loss'] = float(running_loss_bl / len(test_ds_bl))
print_eval_block('BiLSTM - Test Metrics', metrics_bl, y_true_bl, y_pred_bl)

---
## 4. RuBERT (DeepPavlov)

**Type:** Pretrained transformer (BERT)  
**Architecture:** `DeepPavlov/rubert-base-cased` + classifier head  
**Tokenization:** HuggingFace AutoTokenizer (BPE)  
**Preprocessing:** collapse spaces (no lowercase — BERT cased)  
**Checkpoint:** `RuBERT_dusha_resd_train_model.pt` + `_tokenizer/`

In [ ]:
from transformers import AutoTokenizer
import re
from torch import nn
from torch.utils.data import DataLoader, Dataset as TorchDataset

from ruintona.my_experiments.text_models.transformers.RuBERT import EmotionClassifier


class TransformerEmotionDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        enc = tokenizer(
            texts, padding='max_length', truncation=True,
            max_length=max_len, return_tensors='pt',
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attention_mask[idx], self.labels[idx]


checkpoint_bert = load_pytorch_model(
    'dusha_resd_train', models_dir=CHECKPOINTS_DIR,
    model_name='RuBERT', map_location=DEVICE
)

model_params_bert = checkpoint_bert.get('model_params', {})
max_len_bert = int(model_params_bert.get('max_len', 128))
backbone_name = model_params_bert.get('backbone_name', 'DeepPavlov/rubert-base-cased')
dropout_bert = model_params_bert.get('dropout', 0.1)
classifier_hidden = model_params_bert.get('classifier_hidden_size', None)

model_bert = EmotionClassifier(
    model_name=backbone_name, num_classes=4,
    dropout=dropout_bert, classifier_hidden_size=classifier_hidden,
)
model_bert.load_state_dict(checkpoint_bert['model_state_dict'])
model_bert.to(DEVICE)
model_bert.eval()

tokenizer_dir = CHECKPOINTS_DIR / 'RuBERT_dusha_resd_train_tokenizer'
if tokenizer_dir.exists():
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
else:
    tokenizer = AutoTokenizer.from_pretrained(backbone_name)

texts_test_bert, labels_test_str_bert, labels_test_num_bert = load_texts(TEST_LMDB)
# RuBERT uses custom preprocess (keep case, just collapse spaces)
texts_test_bert = [re.sub(r'\s+', ' ', t).strip() for t in texts_test_bert]

test_ds_bert = TransformerEmotionDataset(
    texts_test_bert, labels_test_num_bert, tokenizer, max_len_bert
)
test_loader_bert = DataLoader(test_ds_bert, batch_size=32, shuffle=False)

criterion = nn.CrossEntropyLoss()
all_preds_bert, all_targets_bert, all_probs_bert = [], [], []
running_loss_bert = 0.0
with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader_bert:
        input_ids, attention_mask, labels = (
            input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        )
        logits = model_bert(input_ids, attention_mask)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        running_loss_bert += loss.item() * input_ids.size(0)
        all_preds_bert.append(preds.cpu().numpy())
        all_targets_bert.append(labels.cpu().numpy())
        all_probs_bert.append(probs.cpu().numpy())

y_pred_bert = np.concatenate(all_preds_bert)
y_true_bert = np.concatenate(all_targets_bert)
probs_bert = np.concatenate(all_probs_bert)
metrics_bert = compute_classification_metrics(y_true_bert, y_pred_bert, probs_bert)
metrics_bert['loss'] = float(running_loss_bert / len(test_ds_bert))
print_eval_block('RuBERT - Test Metrics', metrics_bert, y_true_bert, y_pred_bert)